In [8]:
import numpy as np
import polars as pd 

In [12]:
import polars as pl

df = pd.read_csv('03.BaseDPEvolucaoMensalCisp.csv', separator=';', encoding='iso-8859-1')

df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

df = df.drop_nulls(['cisp', 'regiao', 'ano', 'roubo_veiculo'])

# calcular quartis 
q1 = df['roubo_veiculo'].quantile(0.25)
q3 = df['roubo_veiculo'].quantile(0.75)

# calcular limites para outliers
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
lim_inf = q1 - 1.5 * iqr

df_outliers_sup = df.filter(pl.col('roubo_veiculo') > lim_sup).with_columns(
    pl.lit('outliers_sup').alias('flag')
)
df_outliers_inf = df.filter(pl.col('roubo_veiculo') < lim_inf).with_columns(
    pl.lit('outliers_inf').alias('flag')
)

resultado_outliers = pl.concat([df_outliers_sup, df_outliers_inf])
resultado_outliers.write_csv('dp_outliers.csv', separator=';')